I want to split the 1000 genome samples for which I have short read data and assemblies into a train and test set in the ratio of 80/20. Then I want to run the *k*-mer generation, kmer lookup and model training process only based on the train samples and their haplotypes. In the model training notebook I can then also predict how well this works for the test samples.

In my opionion this is an OK tradeoff: Clustering is still based on all samples (to keep the clusters stable), but *k*-mer generation and model building relies only on the test part of the samples. This should give a suitable estimate of the generalization capabilities of HOROSCOPE.

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
### samplesheet with active centromeres, based on chromosome
df = pd.read_csv("/g/korbel/hain/centromere/samplesheets/centromeres_full.tsv", sep="\t")
samples_asm = df[df["ACTIVE"] == True]["SAMPLE"].unique().tolist()

In [4]:
samples_1kgp = [x.split(".")[0] for x in os.listdir("/scratch/hain/centromere/1kgp_unitigs")]

In [5]:
samples_intersect = np.intersect1d(samples_1kgp, samples_asm)

In [6]:
print(f"Samples with assemblies: {len(samples_asm)}")
print(f"1KGP samples: {len(samples_1kgp)}")
print(f"Intersecting samples: {len(samples_intersect)}")

Samples with assemblies: 282
1KGP samples: 272
Intersecting samples: 272


In [9]:
for seed in [10, 21, 31]:
    
    rng = np.random.default_rng(seed=seed)
    
    ### Extract train and test samples from the intersecting samples
    test_samples = rng.choice(
        np.asarray(samples_intersect),
        size=int(len(samples_intersect) * 0.25),
        replace=False,
    ).tolist()
    train_samples = np.setdiff1d(samples_intersect, test_samples)
    
    print(f"Train samples: {len(train_samples)}")
    print(f"Test samples: {len(test_samples)}")
    
    ### write the sets to file to use in other processes

    with open(f"train_samples_1kGP_seed_{seed}.txt", "w") as f:
        for sample in train_samples:
            f.write(f"{sample}\n")
    with open(f"test_samples_1kGP_seed_{seed}.txt", "w") as f:
        for sample in test_samples:
            f.write(f"{sample}\n")

Train samples: 204
Test samples: 68
Train samples: 204
Test samples: 68
Train samples: 204
Test samples: 68
